# Noise–Geometry Phase Diagram for H$_2$ on Neutral-Atom Hardware

**Chemistry → Geometry → Noise → Validity** 📐

This notebook turns the first two workflows into a single summary question:

> **Where does a neutral-atom quantum chemistry workflow remain both low-error and physically valid?**

We use a simple phase-diagram view over:

- **atom spacing** (geometry)
- **noise** $\gamma$

and evaluate a compact quality metric derived from:

- mapped energy
- ground-state reference
- constraint-based validation


## 1. Imports

We again use lightweight scientific Python packages so the notebook remains easy to run and easy to inspect.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt


## 2. Minimal H$_2$ Hamiltonian

We keep the same toy 2-qubit H$_2$ Hamiltonian used in the earlier notebooks so that the new result is purely about **combining geometry, noise, and validation into one operating map**.


In [ ]:
H2_TERMS = [
    ("II", -1.0523732458),
    ("ZI",  0.3979374248),
    ("IZ", -0.3979374248),
    ("ZZ", -0.0112801043),
    ("XX",  0.1809311998),
]

H2_TERMS


## 3. Pauli operators and energy evaluation

We define the 2-qubit operator machinery needed for exact expectation values.


In [ ]:
I = np.array([[1, 0], [0, 1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

PAULI = {
    "I": I,
    "X": X,
    "Y": Y,
    "Z": Z,
}

def kron2(a, b):
    return np.kron(a, b)

def pauli_string_matrix(label: str) -> np.ndarray:
    if len(label) != 2:
        raise ValueError("This notebook expects 2-qubit Pauli strings.")
    return kron2(PAULI[label[0]], PAULI[label[1]])

def hamiltonian_matrix(terms) -> np.ndarray:
    H = np.zeros((4, 4), dtype=complex)
    for label, coeff in terms:
        H = H + coeff * pauli_string_matrix(label)
    return H

def expectation_value(state: np.ndarray, operator: np.ndarray) -> float:
    value = np.vdot(state, operator @ state)
    return float(np.real_if_close(value))

H = hamiltonian_matrix(H2_TERMS)
ground_energy = float(np.min(np.linalg.eigvalsh(H)))

print("Toy Hamiltonian ground-state energy:", ground_energy)
H


## 4. Variational ansatz

We reuse the same small hardware-friendly ansatz from the previous notebooks:

1. start from $|00\rangle$
2. apply single-qubit $R_Y$ rotations
3. apply a CNOT entangling gate
4. apply another layer of $R_Y$ rotations


In [ ]:
def ry(theta: float) -> np.ndarray:
    return np.array([
        [np.cos(theta / 2), -np.sin(theta / 2)],
        [np.sin(theta / 2),  np.cos(theta / 2)],
    ], dtype=complex)

def cnot_01() -> np.ndarray:
    return np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0],
    ], dtype=complex)

def ansatz_state(params) -> np.ndarray:
    theta0, theta1, theta2, theta3 = params

    psi0 = np.array([1, 0, 0, 0], dtype=complex)
    U1 = np.kron(ry(theta0), ry(theta1))
    U2 = np.kron(ry(theta2), ry(theta3))
    Uent = cnot_01()

    psi = U2 @ (Uent @ (U1 @ psi0))
    psi = psi / np.linalg.norm(psi)
    return psi


## 5. Geometry, noise, and validation

We now assemble the same ingredients used in Notebook 02:

- **spacing** and **blockade radius**
- **effective noise** that increases in the strong-blockade regime
- **mapped energy penalty**
- **constraint gate**


In [ ]:
REFERENCE_STATE = np.array([1.0, 0.0, 0.0, 0.0], dtype=complex)
THRESHOLD = 1.0 / np.sqrt(1.0**2 + 1.0**2)

def apply_noise_proxy(state: np.ndarray, gamma: float) -> np.ndarray:
    noisy = state.copy()
    noisy[1:] *= np.exp(-gamma)
    norm = np.linalg.norm(noisy)
    if norm == 0:
        raise ValueError("Noisy state norm is zero.")
    return noisy / norm

def constraint_score(state: np.ndarray, reference: np.ndarray = REFERENCE_STATE) -> float:
    denom = np.linalg.norm(state) * np.linalg.norm(reference)
    if denom == 0:
        raise ValueError("Zero norm encountered in constraint_score.")
    value = np.vdot(reference, state) / denom
    return float(np.real_if_close(value))

def passes_constraint(state: np.ndarray, threshold: float = THRESHOLD) -> bool:
    return constraint_score(state) >= threshold

def blockade_energy_penalty(spacing_um: float, blockade_radius_um: float, scale: float = 0.30) -> float:
    if spacing_um >= blockade_radius_um:
        return 0.0
    strength = 1.0 - (spacing_um / blockade_radius_um)
    return scale * strength

def effective_gamma(gamma: float, spacing_um: float, blockade_radius_um: float, scale: float = 0.80) -> float:
    if spacing_um >= blockade_radius_um:
        return gamma
    strength = 1.0 - (spacing_um / blockade_radius_um)
    return gamma + scale * strength

def energy_of_state(state: np.ndarray) -> float:
    return expectation_value(state, H)

def evaluate_params_gamma_spacing(params, gamma: float, spacing_um: float, blockade_radius_um: float) -> dict:
    gamma_eff = effective_gamma(gamma, spacing_um, blockade_radius_um)
    state = ansatz_state(params)
    noisy_state = apply_noise_proxy(state, gamma_eff)

    raw_energy = energy_of_state(noisy_state)
    penalty = blockade_energy_penalty(spacing_um, blockade_radius_um)
    mapped_energy = raw_energy + penalty

    return {
        "params": np.array(params, dtype=float),
        "gamma": float(gamma),
        "gamma_eff": float(gamma_eff),
        "spacing_um": float(spacing_um),
        "blockade_radius_um": float(blockade_radius_um),
        "raw_energy": float(raw_energy),
        "penalty": float(penalty),
        "energy": float(mapped_energy),
        "constraint_score": constraint_score(noisy_state),
        "passes": passes_constraint(noisy_state),
        "state": noisy_state,
    }


## 6. Sweep spacing and noise

At each point in the $(\gamma, \text{spacing})$ grid, we sample ansatz parameters and record:

- best mapped energy over all states
- best mapped energy among constraint-passing states
- whether any valid state exists

Compared with Notebook 02, we use a moderate sample count to keep the phase boundary visible rather than saturating the valid region.


In [ ]:
rng = np.random.default_rng(17)

gamma_grid = np.linspace(0.0, 1.25, 50)
spacing_grid = np.linspace(3.0, 10.0, 55)
blockade_radius_um = 7.0
n_samples = 120

best_energy_all = np.zeros((len(gamma_grid), len(spacing_grid)))
best_energy_valid = np.full((len(gamma_grid), len(spacing_grid)), np.nan)
valid_exists = np.zeros((len(gamma_grid), len(spacing_grid)), dtype=bool)
best_score_all = np.zeros((len(gamma_grid), len(spacing_grid)))

for i, gamma in enumerate(gamma_grid):
    for j, spacing in enumerate(spacing_grid):
        sampled_params = rng.uniform(0.0, 2.0 * np.pi, size=(n_samples, 4))

        energies = []
        scores = []
        passes = []

        for params in sampled_params:
            result = evaluate_params_gamma_spacing(
                params=params,
                gamma=gamma,
                spacing_um=spacing,
                blockade_radius_um=blockade_radius_um,
            )
            energies.append(result["energy"])
            scores.append(result["constraint_score"])
            passes.append(result["passes"])

        energies = np.array(energies)
        scores = np.array(scores)
        passes = np.array(passes, dtype=bool)

        idx_all = int(np.argmin(energies))
        best_energy_all[i, j] = energies[idx_all]
        best_score_all[i, j] = scores[idx_all]

        if np.any(passes):
            valid_energies = np.where(passes, energies, np.inf)
            idx_valid = int(np.argmin(valid_energies))
            best_energy_valid[i, j] = energies[idx_valid]
            valid_exists[i, j] = True

print("Number of valid cells:", int(np.sum(valid_exists)), "of", valid_exists.size)


## 7. Define phase-diagram quality metrics

We now convert the sweep into simple interpretable quantities.

### Unconstrained energy gap
\[
\Delta E_{\mathrm{all}} = E_{\mathrm{best,all}} - E_0
\]

### Valid energy gap
\[
\Delta E_{\mathrm{valid}} = E_{\mathrm{best,valid}} - E_0
\]

### Quality map
We use the valid energy gap wherever a valid state exists, and leave invalid regions blank.


In [ ]:
gap_all = best_energy_all - ground_energy
gap_valid = best_energy_valid - ground_energy
quality_map = np.where(valid_exists, gap_valid, np.nan)

print("Best unconstrained gap:", np.nanmin(gap_all))
print("Best valid gap:", np.nanmin(gap_valid))


## 8. Main phase diagram: where valid quantum chemistry survives

This is the core figure of the notebook.

Interpretation:

- lower values are better
- blank regions have no valid sampled state
- the viable region shrinks as noise grows and as spacing moves deeper into the blockade regime


In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(
    quality_map,
    aspect="auto",
    origin="lower",
    extent=[spacing_grid.min(), spacing_grid.max(), gamma_grid.min(), gamma_grid.max()],
)
plt.colorbar(label="Valid energy gap from ground state")
plt.xlabel("Atom spacing (μm)")
plt.ylabel("Noise γ")
plt.title("Noise–geometry phase diagram of valid quantum chemistry execution")
plt.tight_layout()
plt.show()


## 9. Comparison phase diagram: unconstrained best gap

This companion plot shows what happens if we ignore the validation gate. Comparing it with the main phase diagram reveals how much the constraint narrows the practical operating window.


In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(
    gap_all,
    aspect="auto",
    origin="lower",
    extent=[spacing_grid.min(), spacing_grid.max(), gamma_grid.min(), gamma_grid.max()],
)
plt.colorbar(label="Unconstrained energy gap from ground state")
plt.xlabel("Atom spacing (μm)")
plt.ylabel("Noise γ")
plt.title("Phase diagram without validity constraint")
plt.tight_layout()
plt.show()


## 10. Validity map

This binary map shows whether at least one sampled state survived the constraint gate at each spacing/noise pair.


In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(
    valid_exists.astype(float),
    aspect="auto",
    origin="lower",
    extent=[spacing_grid.min(), spacing_grid.max(), gamma_grid.min(), gamma_grid.max()],
)
plt.colorbar(label="Any valid state? (1=yes, 0=no)")
plt.xlabel("Atom spacing (μm)")
plt.ylabel("Noise γ")
plt.title("Validity map across spacing and noise")
plt.tight_layout()
plt.show()


## 11. Low-noise and mid-noise slices

A few 1D slices make the phase diagram easier to read:

- low noise
- intermediate noise
- higher noise


In [ ]:
noise_slices = [0.10, 0.45, 0.90]

plt.figure(figsize=(8, 5))
for gamma_target in noise_slices:
    i = int(np.argmin(np.abs(gamma_grid - gamma_target)))
    plt.plot(spacing_grid, quality_map[i], label=f"γ ≈ {gamma_grid[i]:.2f}")

plt.axvline(blockade_radius_um, linestyle="--", label="Blockade radius")
plt.xlabel("Atom spacing (μm)")
plt.ylabel("Valid energy gap")
plt.title("Valid energy gap vs spacing at representative noise levels")
plt.legend()
plt.tight_layout()
plt.show()


## 12. Geometry-limited and noise-limited views

We can also look at the opposite direction: fix spacing and see how the valid gap changes as noise increases.


In [ ]:
spacing_slices = [4.0, 7.0, 9.0]

plt.figure(figsize=(8, 5))
for spacing_target in spacing_slices:
    j = int(np.argmin(np.abs(spacing_grid - spacing_target)))
    plt.plot(gamma_grid, quality_map[:, j], label=f"spacing ≈ {spacing_grid[j]:.1f} μm")

plt.xlabel("Noise γ")
plt.ylabel("Valid energy gap")
plt.title("Valid energy gap vs noise at representative spacings")
plt.legend()
plt.tight_layout()
plt.show()


## 13. Interpretation

This phase diagram shows where quantum chemistry workflows remain both:

- energetically meaningful (low gap to ground state)
- physically valid (pass constraint gate)

The viable region shrinks as:

- noise increases
- spacing enters the blockade regime

This defines a practical operating window for neutral-atom systems.


## 14. Save a figure for the repo

This cell saves the main phase-diagram figure to `../figures/` when run from the `notebooks/` directory inside the repo. The directory is created automatically if needed.


In [ ]:
output_dir = os.path.join("..", "figures")
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "phase_diagram_quality.png")

plt.figure(figsize=(8, 5))
plt.imshow(
    quality_map,
    aspect="auto",
    origin="lower",
    extent=[spacing_grid.min(), spacing_grid.max(), gamma_grid.min(), gamma_grid.max()],
)
plt.colorbar(label="Valid energy gap from ground state")
plt.xlabel("Atom spacing (μm)")
plt.ylabel("Noise γ")
plt.title("Phase diagram: where quantum chemistry remains valid")
plt.tight_layout()
plt.savefig(output_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {output_path}")


## 15. Summary

Notebook 03 turns the earlier results into a single operating-window picture:

- Notebook 01: algorithm + noise + validation
- Notebook 02: algorithm + geometry + blockade
- Notebook 03: where meaningful, valid execution survives

This gives the repository a complete minimal applications story for neutral-atom quantum chemistry workflows.
